In [ ]:
import os
import sys
import yaml
import torch
import jiwer
import gc
import pandas as pd
from tqdm.notebook import tqdm

# Cứu cánh cho Windows khỏi lỗi sập Kernel
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
# Dọn rác VRAM trước khi làm bất cứ việc gì
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# Đảm bảo import được code từ thư mục src
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.recognizer.model import CRNN, ctc_greedy_decode
from src.dataset.loader import OCRDataset, collate_fn
from torch.utils.data import DataLoader
from src.trainer.train_cer import CharsetCodec


In [ ]:

# ==========================================
# 1. CẤU HÌNH FILE CẦN TEST
# ==========================================
CKPT_PATH = '../checkpoints/crnn_best_f3.pth'
MODEL_NAME = 'Phase 3 (F3)'

# ==========================================
# 2. NẠP CONFIG VÀ KHỞI TẠO DATALOADER
# ==========================================
with open('../configs/default.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

target_h = config['preprocess']['target_h']
charset_path = '../' + config['paths']['charset']
val_dir = '../' + config['paths']['synthetic_val']

# Chạy GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[*] Thiết bị suy luận: {device}")

codec = CharsetCodec(charset_path)

val_ds = OCRDataset([val_dir], charset_path, is_train=False, target_h=target_h)
val_loader = DataLoader(
    val_ds, batch_size=64, shuffle=False,
    collate_fn=collate_fn, num_workers=0,
    pin_memory=False # 🔥 QUAN TRỌNG: Phải tắt cái này đi thì Jupyter mới không bị sập Kernel
)

# ==========================================
# 3. NẠP MÔ HÌNH VÀO GPU
# ==========================================
model = CRNN(
    num_classes=len(codec),
    lstm_hidden=config['model']['lstm_hidden'],
    lstm_layers=config['model']['lstm_layers'],
    lstm_dropout=0.0
).to(device)

if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f"Không tìm thấy file: {CKPT_PATH}")

model.load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=True))
model.eval()

# ==========================================
# 4. CHẠY SUY LUẬN & ĐÁNH GIÁ (CER)
# ==========================================
results = []
valid_targets = []
valid_preds = []

for batch in tqdm(val_loader, desc=f'Suy luận {MODEL_NAME}'):
    images = batch['image'].to(device)
    targets = batch['label_str'] 
    
    with torch.no_grad():
        out = model(images)
        preds = ctc_greedy_decode(out, codec.charset)
        
    for t, p in zip(targets, preds):
        results.append({'Nhãn Thực Tế (Target)': t, f'Dự đoán ({MODEL_NAME})': p})
        if len(t.strip()) > 0:
            valid_targets.append(t)
            valid_preds.append(p if len(p.strip()) > 0 else ' ')

# Tính CER
try:
    cer = jiwer.cer(valid_targets, valid_preds) * 100
except Exception:
    cer = 100.0

print(f"\n==========================================")
print(f"🔥 KẾT QUẢ ĐÁNH GIÁ {MODEL_NAME} 🔥")
print(f"==========================================")
print(f"- Số lượng mẫu: {len(results)} ảnh")
print(f"- Lỗi Ký Tự (CER): {cer:.2f}%")
print(f"==========================================\n")

df = pd.DataFrame(results)
display(df.sample(min(15, len(df))))


[*] Mục tiêu đánh giá: Phase 3 (F3) (../checkpoints/crnn_best_f3.pth)
[*] Thiết bị suy luận: cuda
Dataset: 200 samples, train=False
[*] Đã nạp thành công trọng số vào mô hình!


Suy luận Phase 3 (F3):   0%|          | 0/4 [00:00<?, ?it/s]

: 